In [3]:
import pprint as pp
import json
import re
import os
import xmltodict
from deepdiff import DeepDiff
from deepdiff import extract

def print_diffs(left, right):

  def build_item_path(tree_list):
    return('/'.join(str(x) for x in tree_list))

  def build_device_path(tree_list):
    return('root[' + ']['.join(str(x) for x in tree_list) + ']')

  chg = {
    'iterable_item_added': 'ADD',
    'dictionary_item_added': 'ADD',
    'iterable_item_removed': 'DEL',
    'dictionary_item_removed': 'DEL',
    'values_changed': 'CHG'
  }

  dd_tree = DeepDiff(left, right, ignore_order=True, report_repetition=True, view="tree")

  for diff_action in dd_tree:
    print('\n    ' + 25*'.' + diff_action.upper() + 25*'.')
    for item in dd_tree[diff_action]:
      item_path_list = item.path(output_format='list')
      item_path = build_item_path(item_path_list) 
      # device_path_list = item.path(output_format='list')[0:5]
      # device_path =  build_device_path(device_path_list) + '[device-name]'
      print('\n')
      # print(chg[diff_action] + ' | DEVICE:', extract(left, device_path))
      print(chg[diff_action] + ' | PATH  :', item_path)
      print(chg[diff_action] + ' | LEFT  :', item.t1)
      print(chg[diff_action] + ' | RIGHT :', item.t2)


In [21]:
# left_file = 'hardware_inv.json'
# right_file = 'hardware_inv_mod.json'

left_file = 'network-payload-left.json'
right_file = 'network-payload-right.json'

with open(left_file) as json_left:
  left = json.loads(json_left.read())
with open(right_file) as json_right:
  right = json.loads(json_right.read())

print_diffs(left, right)


    .........................VALUES_CHANGED.........................


CHG | PATH  : yang-model-lisko:network/system/description
CHG | LEFT  : Example router
CHG | RIGHT : Example router CHANGED DICT VALUE

    .........................DICTIONARY_ITEM_ADDED.........................


ADD | PATH  : yang-model-lisko:network/router bgp
ADD | LEFT  : not present
ADD | RIGHT : {'asn': '100 NEW DICT'}

    .........................ITERABLE_ITEM_ADDED.........................


ADD | PATH  : yang-model-lisko:network/interfaces/interface/1
ADD | LEFT  : not present
ADD | RIGHT : {'name': 'GigabitEthernet1/1', 'description': 'NEW LIST ITEM interface'}


ADD | PATH  : yang-model-lisko:network/interfaces/interface/3
ADD | LEFT  : not present
ADD | RIGHT : {'NEW DICT in LIST': {'nested1': 'nested1 value1', 'nested2': 'nested2 value2'}}


In [47]:
# We can use the same method for xml comparison

left_file = 'left.xml'
right_file = 'right.xml'

with open(left_file) as xml_left:
  left = json.loads(json.dumps((xmltodict.parse(xml_left.read()))))

with open(right_file) as xml_right:
  right = json.loads(json.dumps((xmltodict.parse(xml_right.read()))))

print_diffs(left, right)



    .........................DICTIONARY_ITEM_REMOVED.........................


DEL | PATH  : config/vrf/vrf-list/1/address-family/ipv4/unicast/export/route-target/address-list/name
DEL | LEFT  : 1:2
DEL | RIGHT : not present

    .........................VALUES_CHANGED.........................


CHG | PATH  : config/vrf/vrf-list/1/rd
CHG | LEFT  : 1:2
CHG | RIGHT : 1:3


CHG | PATH  : config/vrf/vrf-list/1/address-family/ipv4/unicast/import/route-target/address-list/name
CHG | LEFT  : 1:2
CHG | RIGHT : 1:3


CHG | PATH  : config/router/bgp/bgp-no-instance/vrf/1/neighbor/id
CHG | LEFT  : 9.2.2.2
CHG | RIGHT : 9.2.2.3

    .........................DICTIONARY_ITEM_ADDED.........................


ADD | PATH  : config/vrf/vrf-list/1/address-family/ipv4/unicast/export/route-target/address-list/name111
ADD | LEFT  : not present
ADD | RIGHT : 1:2


In [46]:
# Raw output - not easy to read when json is complex..

dd = DeepDiff(left, right, ignore_order=True, report_repetition=True)
pp.pprint(dd, indent=2)

{ 'iterable_item_added': { "root['config']['vrf']['vrf-list'][0]": { 'address-family': { 'ipv4': { 'unicast': { 'export': { 'route-target': { 'address-list': { 'name111': '1:1'}}},
                                                                                                                'import': { 'route-target': { 'address-list': { 'name': '1:1'}}}}}},
                                                                     'name': 'v1',
                                                                     'rd': '*'},
                           "root['config']['vrf']['vrf-list'][1]": { 'address-family': { 'ipv4': { 'unicast': { 'export': { 'route-target': { 'address-list': { 'name': '1:2'}}},
                                                                                                                'import': { 'route-target': { 'address-list': { 'name': '1:3'}}}}}},
                                                                     'name': 'v2',
                                